In [1]:
import pandas as pd
from datasets import load_dataset
from langchain_core.documents import Document

/opt/homebrew/anaconda3/envs/langgraph_v1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Use this ID which is formatted to match HotpotQA schema
dataset_name = "framolfese/2WikiMultihopQA"

try:
    dataset_2wiki = load_dataset(dataset_name, split="validation[:200]")
    print(f"Successfully loaded {dataset_name}")
except Exception as e:
    print(f"Error loading {dataset_name}: {e}")

# Inspect structure to confirm it matches what we expect
if 'dataset_2wiki' in locals():
    print("\nKeys:", dataset_2wiki[0].keys())
    print("\nContext structure:", dataset_2wiki[0]['context'])

Successfully loaded framolfese/2WikiMultihopQA

Keys: dict_keys(['id', 'question', 'answer', 'type', 'evidences', 'supporting_facts', 'context'])

Context structure: {'title': ['Maheen Khan', 'Viktor Yeliseyev', 'Alice Washburn', 'Minamoto no Chikako', 'Polish-Russian War (film)', 'A Snow White Christmas', 'Snow White and the Three Stooges', 'Xawery Żuławski', 'Snow White and the Seven Dwarfs (1955 film)', 'Liberty Ross'], 'sentences': [['Maheen Khan is a Pakistani fashion and costume designer, also an award winner fashion designer for fashion labels like" The Embroidery HouseMaheen" and" Gulabo".', 'She has done many national and international fashion events and shows.', 'She undertook embroidery for the film Snow White and the Huntsman and television series', 'The Jewel in the Crown.'], ['Viktor Petrovich Yeliseyev( born June 9, 1950) is a Russian general, orchestra conductor and music teacher.', 'He is the director of the Ministry of the Interior Ensemble, one of the two Russian Red

In [3]:
dataset_2wiki[0]

{'id': '8813f87c0bdd11eba7f7acde48001122',
 'question': 'Who is the mother of the director of film Polish-Russian War (Film)?',
 'answer': 'Małgorzata Braunek',
 'type': 'compositional',
 'evidences': [['Polish-Russian War', 'director', 'Xawery Żuławski'],
  ['Xawery Żuławski', 'mother', 'Małgorzata Braunek']],
 'supporting_facts': {'title': ['Polish-Russian War (film)',
   'Xawery Żuławski'],
  'sent_id': [1, 2]},
 'context': {'title': ['Maheen Khan',
   'Viktor Yeliseyev',
   'Alice Washburn',
   'Minamoto no Chikako',
   'Polish-Russian War (film)',
   'A Snow White Christmas',
   'Snow White and the Three Stooges',
   'Xawery Żuławski',
   'Snow White and the Seven Dwarfs (1955 film)',
   'Liberty Ross'],
  'sentences': [['Maheen Khan is a Pakistani fashion and costume designer, also an award winner fashion designer for fashion labels like" The Embroidery HouseMaheen" and" Gulabo".',
    'She has done many national and international fashion events and shows.',
    'She undertook em

In [1]:
unique_docs_map = {} # Maps title -> Document

for example in dataset_2wiki:
    # id = example["id"] # Not needed for the doc metadata if we want unique docs across questions
    context = example["context"]
    titles = context["title"]
    sentences_list = context["sentences"]
    
    for title, sentences in zip(titles, sentences_list):
        # Skip if we've already processed this title
        if title in unique_docs_map:
            continue
            
        # Create the chunk text
        chunk_text = f"{title}: " + " ".join(sentences)
        
        # Create Document 
        # Note: We REMOVE 'example_id' from metadata because a single doc 
        # belongs to MANY examples. Including it would make the metadata 
        # unique and force duplication.
        doc = Document(
            page_content=chunk_text, 
            metadata={
                "title": title,
                "dataset": "c" 
            }
        )
        unique_docs_map[title] = doc

# Convert map back to list
documents = list(unique_docs_map.values())

print(f"Created {len(documents)} UNIQUE documents from {len(dataset_2wiki)} examples.")

# Verify
print("\nSample Document:")
print(documents[0].page_content[:200] + "...")
print(documents[0].metadata)

NameError: name 'dataset_2wiki' is not defined

In [ ]:
documents

[Document(metadata={'title': 'Maheen Khan', 'dataset': '2WikiMultihopQA'}, page_content='Maheen Khan: Maheen Khan is a Pakistani fashion and costume designer, also an award winner fashion designer for fashion labels like" The Embroidery HouseMaheen" and" Gulabo". She has done many national and international fashion events and shows. She undertook embroidery for the film Snow White and the Huntsman and television series The Jewel in the Crown.'),
 Document(metadata={'title': 'Viktor Yeliseyev', 'dataset': '2WikiMultihopQA'}, page_content='Viktor Yeliseyev: Viktor Petrovich Yeliseyev( born June 9, 1950) is a Russian general, orchestra conductor and music teacher. He is the director of the Ministry of the Interior Ensemble, one of the two Russian Red Army Choirs.'),
 Document(metadata={'title': 'Alice Washburn', 'dataset': '2WikiMultihopQA'}, page_content='Alice Washburn: Alice Washburn( 1860- 1929) was an American stage and film actress. She worked at the Edison, Vitagraph and Kalem stud

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore


# Initialize the embedding model using your chosen model.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

index_name = "seal-2wiki-v1" 

# Create a Pinecone vector store from the Document objects
vector_store = PineconeVectorStore.from_documents(
    documents,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
pd.DataFrame(dataset_2wiki)


,id,question,answer,type,evidences,supporting_facts,context
0,8813f87c0bdd11eba7f7acde48001122,Who is the mother of the director of film Poli...,Małgorzata Braunek,compositional,"[[Polish-Russian War, director, Xawery Żuławsk...","{'title': ['Polish-Russian War (film)', 'Xawer...","{'title': ['Maheen Khan', 'Viktor Yeliseyev', ..."
1,61a46987092f11ebbdaeac1f6bf848b6,"Which film came out first, Blind Shaft or The ...",The Mask Of Fu Manchu,comparison,"[[Blind Shaft, publication date, 2003], [The M...","{'title': ['Blind Shaft', 'The Mask of Fu Manc...","{'title': ['Blind Shaft', 'The Mysterious Dr. ..."
2,e2a3bf2a0bdd11eba7f7acde48001122,"When did John V, Prince Of Anhalt-Zerbst's fat...",12 June 1516,compositional,"[[John V of Anhalt-Zerbst, father, Ernest I, P...","{'title': ['John V, Prince of Anhalt-Zerbst', ...","{'title': ['Waldemar I, Prince of Anhalt-Zerbs..."
3,0cd3bdea0bde11eba7f7acde48001122,What is the award that the director of film We...,Myanmar Motion Picture Academy Awards,compositional,[[Wearing Velvet Slippers under a Golden Umbre...,{'title': ['Wearing Velvet Slippers under a Go...,"{'title': ['Michael Govan', 'Peter Levin', 'Jo..."
4,f9dcb4a60bda11eba7f7acde48001122,Where was the director of film Ronnie Rocket b...,"Missoula, Montana",compositional,"[[Ronnie Rocket, director, David Lynch], [Davi...","{'title': ['Ronnie Rocket', 'David Lynch'], 's...","{'title': ['Jason Moore (director)', 'Jesse E...."
...,...,...,...,...,...,...,...
195,2cf578340bde11eba7f7acde48001122,When was Ferdinand Iii Of Limburg Stirum's fat...,1701,compositional,"[[Ferdinand III of Limburg Stirum, father, Fer...","{'title': ['Ferdinand III of Limburg Stirum', ...","{'title': ['Ferdinand IV of Limburg Stirum', '..."
196,79675c6e0bd911eba7f7acde48001122,Which country the director of film Ulzhan is f...,German,compositional,"[[Ulzhan, director, Volker Schlöndorff], [Volk...","{'title': ['Ulzhan', 'Volker Schlöndorff'], 's...","{'title': ['John Donatich', 'Peter Levin', 'Ia..."
197,30885e540bb011ebab90acde48001122,Who is Prince Franz De Paula Of Liechtenstein'...,Franz Joseph I,inference,"[[Prince Franz de Paula of Liechtenstein, fath...",{'title': ['Prince Franz de Paula of Liechtens...,"{'title': ['Prince Louis of Liechtenstein', 'P..."
198,6e4854a00bde11eba7f7acde48001122,What is the place of birth of Samir Rifai's fa...,Amman,compositional,"[[Samir Zaid al- Rifai, father, Zaid al-Rifai]...","{'title': ['Samir Rifai', 'Zaid Rifai'], 'sent...","{'title': ['Obata Toramori', 'Zaid Rifai', 'Ol..."


In [ ]:
from langsmith import Client

client = Client()

# Convert to DataFrame for easier handling
df_2wiki = pd.DataFrame(dataset_2wiki)

dataset_name_ls = "2WikiMultihopQA_Validation_200"

if client.has_dataset(dataset_name=dataset_name_ls):
    print(f"Dataset '{dataset_name_ls}' already exists in LangSmith.")
else:
    client.upload_dataframe(
        df=df_2wiki,
        name=dataset_name_ls,
        input_keys=["question", "supporting_facts", "context"],
        output_keys=["answer"],
        description="Validation subset (200 samples) of 2WikiMultiHopQA for SEAL-RAG evaluation."
    )
    print(f"Uploaded dataset '{dataset_name_ls}' to LangSmith.")

Uploaded dataset '2WikiMultihopQA_Validation_200' to LangSmith.
